In [2]:
!pip install numpy matplotlib scipy

  Using cached matplotlib-3.10.8-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (52 kB)
  Using cached scipy-1.17.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 84.3 kB/s  0:01:55m0:00:02m00:04
Using cached matplotlib-3.10.8-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.8 MB)
Using cached scipy-1.17.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
Using cached contourpy-1.3.3-cp314-cp314-m

In [3]:
import numpy as np
from scipy.fft import rfft, rfftfreq
import matplotlib.pyplot as plt

# 1. Setup Simulation Parameters
fs = 44100       # Standard audio sampling rate (Hz)
seconds = 5      # Duration of the recording [cite: 1125]
t = np.linspace(0, seconds, fs * seconds, endpoint=False)

# 2. Create the "Healthy" Baseline (Day 1)
# We'll simulate a 50Hz fundamental hum with a 100Hz harmonic
f_fundamental = 50
healthy_signal = np.sin(2 * np.pi * f_fundamental * t) + 0.5 * np.sin(2 * np.pi * 100 * t)

# 3. Calculate the Fourier Spectrum (Magnitudes)
# rfft returns the complex coefficients; we take the absolute value for magnitude
baseline_fft = np.abs(rfft(healthy_signal))
frequencies = rfftfreq(len(healthy_signal), 1/fs)

In [4]:
# 1. Create a "Faulty" Signal (Day 30)
# We'll add a new 250Hz harmonic to simulate mechanical stress
faulty_signal = healthy_signal + 0.3 * np.sin(2 * np.pi * 250 * t)

# 2. Calculate the "Live" Fourier Spectrum
live_fft = np.abs(rfft(faulty_signal))

# 3. Calculate the Difference (The "Residual")
# We look at the absolute difference between magnitudes at each frequency
fft_diff = np.abs(live_fft - baseline_fft)

# 4. Compute a single "Error Score"
# We'll use the Sum of Squared Errors (SSE) to quantify the total change
error_score = np.sum(fft_diff**2)
print(f"Total Deviation Score: {error_score}")

Total Deviation Score: 1093955625.0


In [5]:
# Calculate the total energy of the baseline (sum of squared magnitudes)
baseline_energy = np.sum(baseline_fft**2)

# Set tolerance as a percentage (e.g., 1%) of the total baseline energy
tolerance_percent = 0.01 
threshold = baseline_energy * tolerance_percent

# Check the health status
if error_score > threshold:
    status = "RED: Potential Fault Detected"
else:
    status = "GREEN: Pump Healthy"

print(f"Threshold: {threshold}")
print(f"Status: {status}")

Threshold: 151938281.25000006
Status: RED: Potential Fault Detected
